## 03_RAG_Wikipedia

----------------------------------

### a) Ziel des Notebooks

1. Mit diesem Notebook soll ein möglichst einfacher RAG_Prozess durchlaufen werden, um die Zusammenhänge und wesentlichen Schritte von klassischem RAG zu verstehen und später zu Agentic RAG auszubauen. Anschließend sollen die Erkenntnisse auf das Projekt GSA - GenSoccerAnalyzer übertragen werden.

2. Es soll außerdem der Zusammenhang zum Gesamtprojekt erarbeitet und verstanden werden. Wie hängen Clustering und RAG zusammen? Wie greifen die Agenten auf die RAG-Struktur zu? Wann und wofür kommt ChromaDB als Vektordatenbank ins Spiel? 

### b) Hinweis
Dieses Notebook wird mithilfe von Gemini und Claude entwickelt.

### c) Ziel von RAG

- Einschränkung der Antworten auf eine Wissensbasis
- Einbezug eines LLM
- Vermeiden von Halluzination


Die RAG-Idee: Statt das Modell aufwendig neu zu trainieren, schickt man bei jeder Frage des Nutzers die passenden Abschnitte aus dem Wikipedia-Artikel automatisch mit. Das LLM liest den Text, den man ihm mitgibt, und formuliert daraus eine präzise Antwort.

----------------------------------------------

## 1. Setup

In [1]:
import wikipediaapi
import pandas as pd

### 1. Schritt: Wikipedia-Artikel FCB über API beziehen

In [2]:
# 1. Wikipedia-API für deutsche Sprache initialisieren
wiki = wikipediaapi.Wikipedia(
    user_agent="GenSoccerAnalyzer/1.0",
    language="de"
)

# 2. Die Seite eines Vereins abrufen
seite = wiki.page("FC Bayern München")

# 3. Prüfen, ob die Seite existiert und Text extrahieren
if seite.exists():
    print(f"Erfolgreich verbunden! Titel: {seite.title}\n")
    
    # Die ersten 500 Zeichen des Haupttextes anzeigen
    print("--- Vorschau des Textes ---")
    print(seite.text[:500] + "...")
    
    # 4. Für späteres Clustering: kompletten Text beziehen
    volltext = seite.text
    print(f"\nGesamtlänge des Artikels: {len(volltext)} Zeichen.")
else:
    print("Seite wurde nicht gefunden. Überprüfe die Schreibweise!")

Erfolgreich verbunden! Titel: FC Bayern München

--- Vorschau des Textes ---
Der Fußball-Club Bayern München e. V., kurz FC Bayern München, Bayern München oder FC Bayern, ist ein deutscher Sportverein aus der bayerischen Landeshauptstadt München. Er wurde am 27. Februar 1900 gegründet und ist mit 432.500 Mitgliedern (Stand: 2. November 2025) ungefähr gleichauf mit Benfica Lissabon einer der beiden mitgliederstärksten Sportvereine der Welt und der mitgliederstärkste Sportverein Deutschlands.
Bekannt wurde der FC Bayern München durch seine Fußballabteilung, die seit 2001 t...

Gesamtlänge des Artikels: 173583 Zeichen.


------------------------------------

### 2. Schritt: Text in Chunks aufteilen

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_text(volltext)
print(f"{len(chunks)} Chunks erstellt")
print("\n--- Beispiel Chunk ---")
print(chunks[0])

514 Chunks erstellt

--- Beispiel Chunk ---
Der Fußball-Club Bayern München e. V., kurz FC Bayern München, Bayern München oder FC Bayern, ist ein deutscher Sportverein aus der bayerischen Landeshauptstadt München. Er wurde am 27. Februar 1900 gegründet und ist mit 432.500 Mitgliedern (Stand: 2. November 2025) ungefähr gleichauf mit Benfica Lissabon einer der beiden mitgliederstärksten Sportvereine der Welt und der mitgliederstärkste Sportverein Deutschlands.


-------------------------------

### 3. Schritt: Chunks in ChromaDB speichern



In [6]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()

ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="paraphrase-multilingual-mpnet-base-v2"
)

collection = client.create_collection("soccer", embedding_function=ef)

collection.add(
    documents=chunks,
    ids=[f"fcb_{i}" for i in range(len(chunks))],
    metadatas=[{"team": "FC Bayern Munich"} for _ in chunks]
)

print(f"{collection.count()} Chunks gespeichert")

c:\Users\Annette\00_Data_Science\06_NLP_GenAI\socceranalyzer_task2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1940.59it/s]


514 Chunks gespeichert


------------------------

### 4. Schritt: Query stellen (Retrieval)

In [6]:
frage = "Wann wurde Bayern München gegründet?"

ergebnisse = collection.query(
    query_texts=[frage],
    n_results=3
)

for i, doc in enumerate(ergebnisse["documents"][0]):
    print(f"--- Chunk {i+1} ---")
    print(doc)

--- Chunk 1 ---
1976 bis 1979 – Der FC Bayern im Umbruch
--- Chunk 2 ---
Der Fußball-Club Bayern München e. V., kurz FC Bayern München, Bayern München oder FC Bayern, ist ein deutscher Sportverein aus der bayerischen Landeshauptstadt München. Er wurde am 27. Februar 1900 gegründet und ist mit 432.500 Mitgliedern (Stand: 2. November 2025) ungefähr gleichauf mit Benfica Lissabon einer der beiden mitgliederstärksten Sportvereine der Welt und der mitgliederstärkste Sportverein Deutschlands.
--- Chunk 3 ---
man auch das Eishockeyspiel auf. Das erste Spiel der Derbygeschichte zwischen dem FC Bayern und den „Sechzigern“ fand im Jahr 1902 statt. Der FC Bayern gewann das Spiel mit 3:0. Um den Spielbetrieb auszuweiten, beschloss der Verein 1906 den Übertritt zum Münchner Sport-Club, behielt aber eine gewisse Eigenständigkeit bei, was sich in der Bezeichnung „F.A. Bayern im Münchner SC“ (F.A. = Fußballabteilung) ausdrückte. Infolge der Fusion trat Bayern nun in weißen Hemden und roten Hosen an, d

-------------------------------------

### 5. Schritt: Antwort generieren (Generation)

- Kostenloser API-Key von Gemini

In [10]:
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.environ.get("GEMINI_API_KEY")

import google.generativeai as genai
genai.configure(api_key=api_key)

In [13]:
# Diagnose: verfügbare Modelle anzeigen
for m in genai.list_models():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

Hier ist das AUGMENTED: Prompt an Gemini (Chunks (Kontext) werden der Frage zugeordnet)

Ohne RAG würde Gemini nur aus seinem Trainingswissen antworten. Durch das Augmenting zwingt man das Modell, sich auf die spezifischen Daten zu stützen — das ist der eigentliche Kern von RAG und auch der Grund warum "Antworte nur basierend auf dem gegebenen Kontext" im System-Prompt steht.

In [14]:
model = genai.GenerativeModel("gemini-2.5-flash")

antwort = model.generate_content(
    f"Du bist ein Fußball-Analyst. Antworte auf Deutsch, "
    f"nur basierend auf dem gegebenen Kontext.\n\n"
    f"Kontext:\n{kontext}\n\n"
    f"Frage: {frage}"
)

print(antwort.text)

Basierend auf dem gegebenen Kontext wurde der FC Bayern München am 27. Februar 1900 gegründet.


--------------------------------------

```
R  →  Retrieval      Schritt 3: ChromaDB liefert die 
                     relevantesten Chunks zur Frage

A  →  Augmented      Der Prompt an Gemini:
                     "Kontext:\n{kontext}\n\nFrage: {frage}"
                     ← hier steckst du die Chunks
                     in die Frage hinein

G  →  Generation     Gemini generiert die Antwort
                     basierend auf dem angereicherten Prompt
```

-----------------------------------

### 6. Schritt: Pipeline bauen (wie später im Projekt GSA)

siehe Ordner Pipeline_RAG_Übung
Starten mit `cd pipeline_uebung`
`uv run python setup/build_database.py`

`TEAMS = [   
    "FC Bayern Munich",   
    "Borussia Dortmund",   
    "Bayer Leverkusen"   
]` 

Pipeline testen: 
`uv run python -m app.run_query` (`-m`damit Python den Ordner findet)

```
build_database.py
        │
        │  1. Wikipedia-Artikel laden (FCB + 2 weitere)
        │  2. Text in Chunks zerschneiden
        │  3. Chunks als Vektoren in ChromaDB speichern
        │
        ▼
chroma_db/ (fertige Datenbank)


run_query.py
        │
        │  1. Frage kommt rein
        │        │
        │        ▼
        │  retriever.py
        │  Frage → Vektor (sentence-transformer)
        │  ChromaDB findet ähnliche Chunks
        │        │
        │        ▼
        │  generator.py
        │  Kontext + Frage → Gemini → Antwort
        │
        ▼
Antwort auf Deutsch
```

### Transformer

Der Transformer (sentence-transformers) ist nicht für die Antwortgenerierung zuständig — er wird zweimal genutzt:
- Beim Aufbau:   Chunks → Vektoren → ChromaDB speichert
- Bei der Query: Frage  → Vektor   → ChromaDB vergleicht
Gemini macht ausschließlich die Textgenerierung. `sentence-transformers` macht ausschließlich die Vektorisierung für die Suche. Das sind zwei komplett verschiedene Aufgaben mit zwei verschiedenen Modellen.

--------------------------------

### 7. Schritt: Deployment über Streamlit Server (kostenlos) 

Verbindung mit Github: https://github.com/anettk404/05_sandbox_task_2
Link: https://gensocceranalyzer2.streamlit.app/

Jedes Mal, wenn man eine Frage ins Chatfenster eingibt, müsste erst die .py-Datei `build_database` gestartet werden  - manuell aus dem Terminal um die ChromaDB zu erstellen. 

#### Problem:     

```
Lokaler Rechner          Streamlit Cloud
      │                        │
chroma_db/ ✓           chroma_db/ ✗
(liegt lokal)          (in .gitignore,
                        nie hochgeladen)
```

#### Lösungen: 

- **Option 1: Datenbank beim App-Start aufbauen (einfachste Lösung)** 
`@st.cache_resource` ist der Schlüssel — Streamlit führt die Funktion nur einmal aus und cached das Ergebnis. Beim ersten Aufruf baut sie die DB auf (~5 Minuten), danach ist sie sofort verfügbar. 

In [ ]:
# streamlit_app.py — ganz oben
# import os
#
#@st.cache_resource  # läuft nur einmal, nicht bei jedem Reload
#def init_pipeline():
    # Datenbank aufbauen falls sie nicht existiert
    #if not os.path.exists("./chroma_db"):
    #from setup.build_database import build
    #build()
    #from pipeline.retriever import get_context
    #from pipeline.generator import generate_answer
    #return get_context, generate_answer

#get_context, generate_answer = init_pipeline()

- **Option 2: Vektordatenbank als Cloud-Service (professionelle Lösung)**

Statt lokaler ChromaDB einen gehosteten Service nutzen: Pinecone, Weaviate, Qdrant. Die Datenbank läuft permanent in der Cloud, kein manueller Aufbau nötig.    


- **Option 3: DB als Datei ins Repository (pragmatisch für kleine Projekte)**

```
# .gitignore anpassen:
# chroma_db/    ← diese Zeile entfernen oder auskommentieren
```
Dann die DB committen — funktioniert solange sie klein ist (~50MB Limit bei GitHub).
